In [ ]:
# Helper Script: Load Seurat object and transform to anndata for input for analysis workflow

# Prerequisite - Load Libraries

In [ ]:
import h5py
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix

In [ ]:
import os

In [ ]:
### Check the path of loaded packages
sc.__path__

# Preqrequisites Configurations & Parameters

In [ ]:
### Load the parameters that are set via the configuration files

In [ ]:
### Load configurations file
global_configs = pd.read_csv('configurations/Data_Configs.csv', sep = ',')

In [ ]:
data_path = global_configs['value'][global_configs['parameter'] == 'data_path']

In [ ]:
data_path

In [ ]:
result_path = global_configs['value'][global_configs['parameter'] == 'result_path']

In [ ]:
result_path

In [ ]:
### Data name of sc dataset

In [ ]:
files = [f for f in os.listdir(data_path[0]) if os.path.isfile(os.path.join(data_path[0], f))]

In [ ]:
files

In [ ]:
### Get only the seurat files to convert

In [ ]:
# Filter the list to include only files with '.h5seurat' in the name
h5seurat_files = [f for f in files if '.h5seurat' in f]

In [ ]:
h5seurat_files = [f.replace('.h5seurat', '') for f in h5seurat_files]

In [ ]:
h5seurat_files

# Conversion

In [ ]:
### Load single-cell datasets in seurat and convert to anndata

In [ ]:
# Open the .h5seurat file

for i in h5seurat_files:
    file_path = os.path.join(data_path[0], f'{i}'+ '.h5seurat')
    output_path = os.path.join(data_path[0], f'{i}'+ '.h5ad')

    with h5py.File(file_path, 'r') as f:
        # Access the counts group
        counts_group = f['assays/RNA/counts']  ## this needs to exist in seurat structure otherwise conversion won't work; see structure below
    
        # Create the sparse matrix
        data_array = counts_group['data'][:]
        indices_array = counts_group['indices'][:]
        indptr_array = counts_group['indptr'][:]
        shape = (len(indptr_array) - 1, indices_array.max() + 1)
    
        data = csr_matrix((data_array, indices_array, indptr_array), shape=shape)
    
        # Extract gene names and cell barcodes
        genes = [x.decode('utf-8') for x in f['assays/RNA/meta.features/_index'][:]] ## this needs to exist in seurat structure otherwise conversion won't work; see structure below
        barcodes = [x.decode('utf-8') for x in f['cell.names'][:]] ## this needs to exist in seurat structure otherwise conversion won't work; see structure below

        # Extract metadata if available
        metadata = {}
        if 'meta.data' in f:
            for key in f['meta.data'].keys():
                metadata[key] = [x.decode('utf-8') if isinstance(x, bytes) else x for x in f['meta.data'][key][:]]
            metadata = pd.DataFrame(metadata, index=barcodes)

    # Create AnnData object
    adata = AnnData(X=data, var=pd.DataFrame(index=genes), obs=pd.DataFrame(index=barcodes))

    # Add metadata to AnnData object if available
    if not metadata.empty:
        del metadata['_index']
        adata.obs = metadata

    # Optionally save to an .h5ad file
    adata.write( output_path)

In [ ]:
pd.crosstab(adata.obs['cluster_id'], adata.obs['cluster_id']).sum(axis=1)

# Inspection of h5seurat file (use if structure is different from the one assumed above)

In [ ]:
# Open the .h5seurat file

for i in h5seurat_files:
    file_path = os.path.join(data_path[0], f'{i}'+ '.h5seurat')
    with h5py.File(file_path, 'r') as f:
        # Function to recursively print the structure of the HDF5 file
        def print_hdf5_structure(name, obj):
            if isinstance(obj, h5py.Group):
                print(f"Group: {name}")
            elif isinstance(obj, h5py.Dataset):
                print(f"Dataset: {name}, shape: {obj.shape}, dtype: {obj.dtype}")

        # Print the structure of the HDF5 file
        f.visititems(print_hdf5_structure)